In [1]:
import numpy as np
import nbimporter
import cepairsimplementation as ce
from reci import RECI
from lingam import lingam
from igci import IGCI
from anm import anm
from pnl import pnl
from cgnn import cgnn
from emd import emd
import os
import glob
import matplotlib.pyplot as plt
import traceback
import functools
from scipy.optimize import nnls
import pandas as pd
from sklearn.linear_model import Lasso, Ridge
import json
import math
import csv
import synthetic_nn_keras as synth_k
import re
import cvxpy as cp
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
def getSynthetic(dir):
    data_list = []
    txt_files = glob.glob(os.path.join(dir, '*.txt'))
    
    for file_path in txt_files:
        # Load numerical data from text file into a numpy array
        data = np.loadtxt(file_path)
        
        # Verify there are exactly 2 columns
        if data.ndim != 2 or data.shape[1] != 2:
            continue  # Skip files with invalid format
        
        # Extract columns and convert to Python lists
        x = np.array(data[:, 0]).reshape(-1, 1)
        y = np.array(data[:, 1]).reshape(-1, 1)
        
        data_list.append([x,y])
    return data_list


def getOld(dataset):
    folder="../other implementations/synthetic_datasets"
    pairs_file   = f"{folder}/{dataset}_pairs.csv"
    targets_file = f"{folder}/{dataset}_targets.csv"
    
    # --- sanity checks ---
    if not os.path.isfile(pairs_file):
        raise FileNotFoundError(f"Pairs file not found: {pairs_file}")
    if not os.path.isfile(targets_file):
        raise FileNotFoundError(f"Targets file not found: {targets_file}")
    
    # --- read in with pandas ---
    df_pairs   = pd.read_csv(pairs_file)
    df_targets = pd.read_csv(targets_file)
    
    if len(df_pairs) != len(df_targets):
        raise ValueError(
            f"Row count mismatch: {len(df_pairs)} in pairs vs "
            f"{len(df_targets)} in targets"
        )
    
    data_list = []
    for idx, pair_row in df_pairs.iterrows():
        # get the raw space‐separated strings
        x_str = str(pair_row.iloc[1])
        y_str = str(pair_row.iloc[2])
        
        # split & convert to floats, then make column vectors
        x = np.array([float(v) for v in x_str.split()]).reshape(-1, 1)
        y = np.array([float(v) for v in y_str.split()]).reshape(-1, 1)
        
        # swap if target's 2nd column is -1
        if df_targets.iloc[idx, 1] == -1:
            x, y = y, x
        data_list.append([x, y])
    
    return data_list

def save_weights(weights, weight_labels, metrics, filename='data/dataset_weights.csv'):
    """
    Save a matrix of weights with labels and metric names to a CSV file.

    Parameters
    ----------
    weights : sequence of sequence of float
        A 2D sequence where each row corresponds to a weight and each column to a metric.
    weight_labels : sequence of str
        Human-readable names for each weight (one per row).
    metrics : sequence of str
        Names of the metrics (one per column).
    filename : str, optional
        Path to the output CSV file (default is 'data/dataset_weights.csv').

    Raises
    ------
    ValueError
        If the number of rows in weights does not match weight_labels,
        or the number of columns in weights does not match metrics.
    """
    # Check dimensions
    n_rows = len(weights)
    if n_rows != len(weight_labels):
        raise ValueError(f"Number of weight rows ({n_rows}) does not match number of labels ({len(weight_labels)})")
    # Ensure each row has the same number of metrics
    if n_rows > 0:
        n_cols = len(weights[0])
        if n_cols != len(metrics):
            raise ValueError(f"Number of weight columns ({n_cols}) does not match number of metrics ({len(metrics)})")
        for row in weights:
            if len(row) != n_cols:
                raise ValueError("All rows in weights must have the same length")
    
    # Write to CSV
    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        # write header: first column is 'label', then metric names
        writer.writerow(['label'] + list(metrics))
        # write each label and its metrics
        for label, row in zip(weight_labels, weights):
            writer.writerow([label] + list(row))

    print(f"Saved {n_rows} weight entries with {len(metrics)} metrics to '{filename}'.")


In [3]:
def run_synthetic(func,dataset, **kwargs):
    if dataset[:2] == "CE":
        data = getOld(dataset)
    else:
        dir=f"generate_txt/{dataset}"
        data = getSynthetic(dir)
    for d in data:
        if np.isnan(d).any():
            print("Found NaN in data:", dataset, d)
    weights=np.ones(len(data))/len(data)
    scores =np.array([func(d,**kwargs) for d in data])
    kwargs_str = ", ".join(f"{k}={v}" for k, v in kwargs.items())
    folder=f"predictions/{func.__name__}_{kwargs_str}"
    os.makedirs(folder, exist_ok=True)
    np.savetxt(f"{folder}/{dataset}.txt", np.column_stack((scores, weights)))

def analyse_predictions(funcname,dataset, **kwargs):
    kwargs_str = ", ".join(f"{k}={v}" for k, v in kwargs.items())
    folder=f"predictions/{funcname}_{kwargs_str}"
    data = np.loadtxt(f"{folder}/{dataset}.txt")
    # Split the data into scores and weights
    scores = data[:, 0]  # First column
    weights = data[:, 1]  # Second column
    
    #AUROC
    y_scores, y_true = ce.switch_signs(scores)
    normalized_y=ce.minmax_scale(y_scores)
    normalized_y[np.isnan(normalized_y)] = 0.5
    auroc = ce.roc_auc_score(y_true,normalized_y,sample_weight=weights)
    #Accuracy
    guess=ce.sign_to_binary(scores)<0
    accuracy = sum(guess*weights)/sum(weights)


    #plot
    ce.save_excel(funcname,dataset,[auroc,accuracy],["Auroc","Accuracy"])


In [4]:
def test_all_synthetic(method):
    # Path to the parent folder
    parent_folder = "generate_txt/"
    # Loop over the folders inside the parent folder
    for dataset in os.listdir(parent_folder):
        if "v3" not in dataset:
            continue
        #chock if noise is true
        run_synthetic(method,dataset)
        analyse_predictions(method.__name__,dataset)

def test_all_old(method):
    # Path to the parent folder
    datasets=["CE-Gauss","CE-Net","CE-Multi"]
    # Loop over the folders inside the parent folder
    for dataset in datasets:
        run_synthetic(method,dataset)
        analyse_predictions(method.__name__,dataset)

def trace_wrapper(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print("Traceback for function call:")
        traceback.print_stack()  # Print the call stack to standard output.
        return fn(*args, **kwargs)
    return wrapper

def testRsynthetic(method_name,**kwargs):
    # Path to the parent folder
    parent_folder = "generate_txt/"
    # Loop over the folders inside the parent folder
    for dataset in os.listdir(parent_folder):
        analyse_predictions(method_name,dataset)

In [6]:
test_all_old(RECI)

In [5]:
analyse_predictions("qcd_functionR","CE-Cha")

In [10]:
def compute_full_metric(method: str,
                   weights: np.ndarray,
                   labels: list[str],
                   metric: str,
                   predictions_dir: str = 'predictions'
                  ) -> float:
    """
    Compute a weighted metric (AUROC or accuracy) for a given prediction method.

    Parameters
    ----------
    method : str
        Subdirectory under `predictions_dir` containing per-label .txt files.
    weights : np.ndarray
        Array of per-label weights to use when computing the chosen metric.
    labels : list of str
        List of label names corresponding to rows in `weights`.
    metric : {'auroc', 'accuracy'}
        Which metric to compute.
    predictions_dir : str, optional
        Base directory where method subfolders live (default: 'predictions').

    Returns
    -------
    float
        The weighted metric value.

    Raises
    ------
    ValueError
        If `metric` is not one of the supported options.
    """
    full_scores = []
    sample_weights = []
    a=[]
    guessesv2 = []

    method_dir = os.path.join(predictions_dir, method.split(' ')[0] + '_')
    for label_w, label in zip(weights, labels):
        file_path = os.path.join(method_dir, f"{label}.txt")
        if not os.path.exists(file_path):
            continue

        data = np.loadtxt(file_path)
        scores = data[:, 0]
        inst_w  = data[:, 1]  # per-instance weights in file

        full_scores.extend(scores)
        sample_weights.extend(label_w * inst_w)
    if not full_scores:
        raise ValueError(f"No prediction files found for method '{method}'")

    full_scores    = np.array(full_scores)
    sample_weights = np.array(sample_weights)
    if metric == 'Auroc':
        # Prepare true labels and normalize scores
        y_scores, y_true = ce.switch_signs(full_scores)
        y_norm = ce.minmax_scale(y_scores)
        y_norm[np.isnan(y_norm)] = 0.5
        return ce.roc_auc_score(y_true, y_norm, sample_weight=sample_weights)

    elif metric == 'Accuracy':
        guess = ce.sign_to_binary(full_scores)
        return np.sum(guess * sample_weights) / np.sum(sample_weights)
    else:
        print(f"Unsupported metric: {metric}")
        raise ValueError("Unsupported metric: choose 'auroc' or 'accuracy'")

In [12]:
def compute_matrices(
    methods=["RECI","IGCI","lingam","anm","pnl","emd","cgnn","SlopeR","qcd_functionR"],
    metrics=["Auroc","Accuracy"],
    filepath="../previous_results.xlsx",
    noisy="no"
):
    """
    Process an Excel file and create matrices for a linear regression formulation Ax - b = 0,
    but only keep those exchangeable datasets tested by every method-variant.

    Parameters:
      methods: list of strings
          Method names (from column 1 of the Excel file) to include.
      metrics: list of strings
          The names of the metric columns to extract.
      filepath: string
          Path to the Excel file.
      noisy: string, one of {"All", "yes", "no"}
          Controls inclusion of noisy datasets (datasets whose name contains "noise=True").
            - "All": keep both noisy and non-noisy datasets.
            - "yes": keep only noisy datasets.
            - "no": exclude noisy datasets (default).

    Returns:
      A: numpy.ndarray (3D array)
          Shape (n_method_variants, n_common_datasets, n_metrics)
      b: numpy.ndarray (2D array)
          Shape (n_method_variants, n_metrics)
      weights: numpy.ndarray (1D array)
          Relative weight for each method variant (each base method sums to 1).
      labels_dim1: list of strings
          Method‑variant labels ("Method (Parameters)").
      labels_dim2: list of strings
          The filtered exchangeable dataset names (common to all variants, respecting noisy setting).
      labels_dim3: list of strings
          The metric names.
    """
    # Read the Excel file
    df_all = pd.read_excel(filepath)

    # Keep only relevant columns
    required_cols = ['Method', 'Dataset', 'Parameters'] + metrics
    df_all = df_all[required_cols]

    # Filter by requested methods
    df_all = df_all[df_all['Method'].isin(methods)]

    # Build method‑variant label on full data
    df_all['Method_Var'] = df_all['Method'].astype(str) + " (" + df_all['Parameters'].astype(str) + ")"
    method_variants = df_all['Method_Var'].unique().tolist()
    labels_dim1 = method_variants
    n_methods = len(method_variants)

    # --- Compute b from 'tuebingen', ignoring noisy filter ---
    b = np.zeros((n_methods, len(metrics)))
    for i, mvar in enumerate(method_variants):
        sub_all = df_all[df_all['Method_Var'] == mvar]
        tu = sub_all[sub_all['Dataset'].str.lower() == 'tuebingen']
        if not tu.empty:
            b[i, :] = tu[metrics].mean(axis=0).values

    # --- Prepare df for A by applying noisy filter ---
    df = df_all.copy()
    noisy_opt = noisy.lower()
    if noisy_opt == "no":
        df = df[~df['Dataset'].str.contains('noise=True', case=False, na=False)]
    elif noisy_opt == "yes":
        df = df[df['Dataset'].str.contains('noise=True', case=False, na=False)]
    elif noisy_opt == "all":
        pass
    else:
        raise ValueError("Invalid value for 'noisy': choose 'All', 'yes', or 'no'.")

    # Identify all exchangeable datasets
    exch = df[df['Dataset'].str.contains('exchangeable', case=False, na=False)].copy()

    # Only keep those datasets that appear for every method‑variant
    counts = exch.groupby('Dataset')['Method_Var'].nunique()
    common_datasets = counts[counts == n_methods].index.tolist()
    labels_dim2 = common_datasets

    labels_dim3 = metrics
    n_datasets = len(labels_dim2)
    n_metrics = len(metrics)

    # Initialize A
    A = np.zeros((n_methods, n_datasets, n_metrics))

    # Populate A
    for i, mvar in enumerate(method_variants):
        sub = df[df['Method_Var'] == mvar]
        for j, ds in enumerate(labels_dim2):
            sel = sub[sub['Dataset'] == ds]
            if not sel.empty:
                A[i, j, :] = sel[metrics].mean(axis=0).values

    # Compute per‑variant weights so each base method sums to 1
    method_counts = df_all.groupby('Method')['Method_Var'].nunique().to_dict()
    weights = np.array([
        1.0 / method_counts.get(mvar.split(' (')[0], 1)
        for mvar in method_variants
    ])

    return A, b, weights, labels_dim1, labels_dim2, labels_dim3

def compute_coefficients_general(A, b, weights, regression_type='nnls', alpha=1.0, penalty='l2'):
    n_methods, n_datasets, n_metrics = A.shape
    c = np.zeros((n_datasets, n_metrics))
    errors = np.zeros(n_metrics)

    W = np.diag(weights)
    for k in range(n_metrics):
        A_k = A[:, :, k]
        b_k = b[:, k]

        A_w = W @ A_k
        b_w = W @ b_k

        if np.any(np.isnan(A_w)) or np.any(np.isnan(b_w)):
            raise ValueError(f"NaN values found in weighted matrices for metric {k}.")
        if np.any(np.isinf(A_w)) or np.any(np.isinf(b_w)):
            raise ValueError(f"Inf values found in weighted matrices for metric {k}.")

        n = A_w.shape[1]
        x = cp.Variable(n, nonneg=True) 
        constraints = [cp.sum(x) == 1]

        if regression_type == 'nnls':
            obj = cp.sum_squares(A_w @ x - b_w)
        elif regression_type == 'ridge':
            obj = 0.5 * cp.sum_squares(A_w @ x - b_w) + alpha * cp.sum_squares(x)
        elif regression_type == 'lasso':
            obj = 0.5 * cp.sum_squares(A_w @ x - b_w) + alpha * cp.norm1(x)
        elif regression_type=="l1norm":
            obj = cp.sum(cp.abs(A_w @ x - b_w)) + alpha * cp.norm1(x)
        else:
            raise ValueError(f"Unsupported regression type: {regression_type}")
        prob = cp.Problem(cp.Minimize(obj), constraints)
        prob.solve(solver=cp.SCS)  # or another solver you have installed
        x_k = x.value 
        c[:, k] = x_k

        # Compute error using matching norm
        predictions = A_k @ x_k
        if penalty == 'l1':
            errors[k] = np.mean(np.abs(predictions - b_k))  # L1 error
        else:
            errors[k] = np.mean((predictions - b_k)**2) # L2 error

    return c, errors


def leave_one_out_validation(A, b, weights, labels1_dim, labels2_dim, labels3_dim,regression_type='nnls', alpha=1.0, penalty='l2'):
    n_methods, n_datasets, n_metrics = A.shape

    # Extract method names
    method_names = [label.split('(')[0].strip() for label in labels1_dim]
    unique_methods = list(dict.fromkeys(method_names))  # preserves order
    n_unique = len(unique_methods)

    error_matrix = np.zeros((n_metrics, n_unique))
    full_matrix = np.zeros((n_metrics, n_unique))

    for i, method in enumerate(unique_methods):
        indices = [j for j, name in enumerate(method_names) if name == method]

        # Create training and testing sets
        mask = np.ones(n_methods, dtype=bool)
        mask[indices] = False

        A_train = A[mask]
        b_train = b[mask]
        weights_train = weights[mask]

        A_test = A[~mask]
        b_test = b[~mask]

        x, _ = compute_coefficients_general(A_train, b_train, weights_train,
                                            regression_type=regression_type, alpha=alpha)
        
        for k in range(n_metrics):
            predictions = A_test[:, :, k] @ x[:, k]
            if penalty == 'l1':
                metric_error = np.mean(np.abs(predictions - b_test[:, k])) # L1
                full_error= np.mean(np.abs(compute_full_metric(method, x[:, k], labels2_dim, labels3_dim[k]) - b_test[:, k])) # L1
            else:
                metric_error = np.mean((predictions - b_test[:, k])**2) # L2
                full_error = np.mean((compute_full_metric(method, x[:, k], labels2_dim, labels3_dim[k]) - b_test[:, k])**2)
            error_matrix[k, i] = metric_error
            full_matrix[k,i]= full_error


    return error_matrix,full_matrix

In [46]:
metrics=["Auroc","Accuracy"]
A, b, weights, labels_dim1, labels_dim2, labels_dim3= compute_matrices(metrics=metrics, noisy="no")

In [47]:
print("b",b[:10,:10])
print(labels_dim1)

b [[0.21146986 0.28521279]
 [0.75600423 0.701966  ]
 [0.70850304 0.65326161]
 [0.5        0.50528362]
 [0.42054354 0.39578782]
 [0.41274164 0.44481113]
 [0.72650445 0.64535468]
 [0.66251142 0.60865329]
 [0.73022902 0.69956587]]
['SlopeR (nan)', 'RECI (nan)', 'IGCI (nan)', 'lingam (nan)', 'anm (nan)', 'pnl (nan)', 'emd (nan)', 'cgnn (nan)', 'qcd_functionR (nan)']


In [48]:
coefs, errors = compute_coefficients_general(A, b, weights,regression_type="ridge",alpha=1)
print(errors)
save_weights(coefs, labels_dim2,metrics)

[0.0014389 0.0013915]
Saved 81 weight entries with 2 metrics to 'data/dataset_weights.csv'.


In [49]:
cross_errors,full_errors=leave_one_out_validation(A, b, weights, labels_dim1,labels_dim2,labels_dim3,regression_type="ridge",alpha=1, penalty='l2')
print("Cross-validation errors:Ridge l2")
print(np.mean(cross_errors,axis=1))
print(np.mean(full_errors,axis=1))

cross_errors,full_errors=leave_one_out_validation(A, b, weights, labels_dim1,labels_dim2,labels_dim3,regression_type="ridge",alpha=1, penalty='l1')
print("Cross-validation errors:Ridge l1")
print(np.mean(cross_errors,axis=1))
print(np.mean(full_errors,axis=1))

Cross-validation errors:Ridge l2
[0.0112648  0.00831835]
[0.01624238 0.00831506]
Cross-validation errors:Ridge l1
[0.06074502 0.04897794]
[0.08407374 0.04892898]


In [50]:
def compute_diff_matrix(table, methods, norm_type='l2'):
    # The datasets (rows) and metrics (columns) we want to report
    datasets = ["CE-Cha", "CE-Net", "CE-Gauss", "CE-Multi"]
    metrics = ["auroc", "accuracy"]
    
    # Initialize an empty result dictionary
    result_matrix = {ds: {} for ds in datasets}
    
    # Loop over datasets and metrics
    for ds in datasets:
        for metric in metrics:
            # Collect difference norms for each method (only if the method exists in the table)
            norm_values = []
            for method in methods:
                # Check that the method exists and has both the dataset and the baseline "CE-Tueb"
                if method in table and ds in table[method] and "CE-Tueb" in table[method]:
                    method_val = table[method][ds][metric]
                    baseline_val = table[method]["CE-Tueb"][metric]
                    diff = method_val - baseline_val
                    # For a scalar, the L1 norm is the absolute difference.
                    if norm_type == 'l1':
                        norm_val = abs(diff)
                    elif norm_type == 'l2':
                        norm_val = diff**2
                    else:
                        raise ValueError("norm_type must be either 'l1' or 'l2'.")
                    norm_values.append(norm_val)
                else:
                    print(f"Warning: Missing data for method '{method}' on dataset '{ds}' or 'CE-Tueb'.")
            # Average the differences across the provided methods.
            if norm_values:
                avg_norm = sum(norm_values) / len(norm_values)
            else:
                avg_norm = None
            result_matrix[ds][metric] = avg_norm
    return result_matrix

# Example usage:
# Load the saved table.
json_filename = "data/book_results.json"
with open(json_filename, "r") as f:
    loaded_table = json.load(f)

# Define the list of methods you want to consider.
# (For instance, you can choose to include all methods or a subset; here we include all available methods.)

methods_to_consider = ["RECI", "LiNGAM", "IGCI", "ANM", "PNL", "CGNN"]


# Compute the difference matrix.
diff_matrix = compute_diff_matrix(loaded_table, methods_to_consider, norm_type="l1")

print(diff_matrix)


{'CE-Cha': {'auroc': 0.11849999999999994, 'accuracy': 0.10766666666666665}, 'CE-Net': {'auroc': 0.12233333333333334, 'accuracy': 0.06749999999999999}, 'CE-Gauss': {'auroc': 0.23183333333333334, 'accuracy': 0.19766666666666666}, 'CE-Multi': {'auroc': 0.24783333333333335, 'accuracy': 0.14150000000000004}}


In [51]:
def analyze_performance(filepath: str, methods: list, metrics: list, compare_datasets: list, norm_type: str = 'l1'):
    """
    Analyzes the performance of methods across different datasets by comparing them
    to a baseline established from the 'tuebingen' dataset.

    Args:
        filepath (str): The path to the Excel file containing the results.
                        Expected columns: 'Method', 'Dataset', 'Parameters', and all `metrics`.
        methods (list): A list of method names (strings) to include in the analysis.
        metrics (list): A list of metric names (strings) for which to perform the analysis.
        compare_datasets (list): A list of dataset names (strings) to compare against
                                 the 'tuebingen' baseline.
        norm_type (str, optional): The type of average difference to compute.
                                   'l1' for mean absolute difference (default).
                                   'l2' for mean squared difference.
                                   Case-insensitive.
    """
    if not os.path.exists(filepath):
        print(f"Error: File not found at {filepath}")
        return

    try:
        df_all = pd.read_excel(filepath)
    except Exception as e:
        print(f"Error reading Excel file: {e}")
        return

    # Ensure required columns exist
    required_base_cols = ['Method', 'Dataset', 'Parameters']
    for col in required_base_cols:
        if col not in df_all.columns:
            print(f"Error: Required column '{col}' not found in the Excel file.")
            return
    for metric in metrics:
        if metric not in df_all.columns:
            print(f"Error: Metric column '{metric}' not found in the Excel file.")
            return

    # Keep only relevant columns
    required_cols = required_base_cols + metrics
    df_all = df_all[required_cols]

    # Filter by requested methods
    # Now filter
    df_all = df_all[df_all['Method'].isin(methods)].copy()
    

    if df_all.empty:
        print("No data found for the specified methods after filtering.")
        return

    # Build method-variant label on full data
    df_all['Method_Var'] = df_all['Method'].astype(str) + " (" + df_all['Parameters'].astype(str) + ")"
    method_variants = df_all['Method_Var'].unique().tolist()
    n_methods = len(method_variants)

    # Map method_variant to its index for easy lookup
    method_var_to_idx = {mvar: i for i, mvar in enumerate(method_variants)}

    # --- Compute b from 'tuebingen', ignoring noisy filter ---
    # b will store the mean performance for each method-variant on 'tuebingen' for each metric
    b = np.zeros((n_methods, len(metrics)))
    print("\n--- Tuebingen Baseline (b) ---")
    for i, mvar in enumerate(method_variants):
        # Case-insensitive matching for 'tuebingen'
        sub_all = df_all[df_all['Method_Var'] == mvar]
        tu = sub_all[sub_all['Dataset'].str.lower() == 'tuebingen']
        if not tu.empty:
            # Calculate mean for each metric for the current method-variant on 'tuebingen'
            b[i, :] = tu[metrics].mean(axis=0).values
            print(f"  {mvar}: {dict(zip(metrics, b[i, :]))}")
        else:
            print(f"  Warning: No 'tuebingen' data found for method variant: {mvar}. Baseline will be 0 for this variant.")

    print("\n--- Average Differences from Tuebingen Baseline ---")
    # Store results for structured printing
    all_results = {}

    for dataset_name in compare_datasets:
        print(f"\nDataset: {dataset_name}")
        # Case-insensitive matching for compare_datasets
        df_dataset = df_all[df_all['Dataset'].str.lower() == dataset_name.lower()]

        if df_dataset.empty:
            print(f"  No data found for dataset '{dataset_name}'. Skipping.")
            continue

        dataset_metric_averages = {}
        for j, metric in enumerate(metrics):
            metric_diffs = []
            for i, mvar in enumerate(method_variants):
                method_data = df_dataset[df_dataset['Method_Var'] == mvar]
                if not method_data.empty:
                    # Calculate mean for the current method-variant on the comparison dataset
                    current_value = method_data[metric].mean()
                    tuebingen_value = b[i, j] # Get the baseline value

                    diff = current_value - tuebingen_value
                    metric_diffs.append(diff)
                else:
                    print(f"  Warning: No data for {mvar} on dataset '{dataset_name}' for metric '{metric}'.")

            if metric_diffs:
                if norm_type.lower() == 'l1':
                    avg_diff = np.mean(np.abs(metric_diffs))
                    print_norm_type = "Mean Absolute Difference (L1 Avg)"
                elif norm_type.lower() == 'l2':
                    avg_diff = np.mean(np.square(metric_diffs))
                    print_norm_type = "Mean Squared Difference (L2 Avg)"
                else:
                    print(f"  Invalid norm_type '{norm_type}'. Using L1 (Mean Absolute Difference).")
                    avg_diff = np.mean(np.abs(metric_diffs))
                    print_norm_type = "Mean Absolute Difference (L1 Avg)"

                print(f"    Metric '{metric}' ({print_norm_type}): {avg_diff:.4f}")
                dataset_metric_averages[metric] = avg_diff
            else:
                print(f"    Metric '{metric}': No sufficient data to compute average difference.")
                dataset_metric_averages[metric] = "N/A"
        all_results[dataset_name] = dataset_metric_averages

    print("\n--- Summary of All Averages ---")
    for dataset, metrics_data in all_results.items():
        print(f"Dataset: {dataset}")
        for metric, avg_value in metrics_data.items():
            print(f"  Metric '{metric}': {avg_value}")

#["RECI", "LiNGAM", "IGCI", "ANM", "PNL", "CGNN"]
#methods=["RECI", "lingam", "IGCI", "anm", "pnl", "cgnn"]
methods=["RECI","IGCI","lingam","anm","pnl","emd","cgnn","SlopeR","qcd_functionR"]
metrics=["Auroc","Accuracy"]

analyze_performance(
    filepath="../previous_results.xlsx",
    methods=methods,
    metrics=metrics,
    compare_datasets=["CE-Cha", "CE-Net", "CE-Gauss", "CE-Multi"],
    norm_type='l1'
)


--- Tuebingen Baseline (b) ---
  SlopeR (nan): {'Auroc': np.float64(0.2114698638600033), 'Accuracy': np.float64(0.2852127935093939)}
  RECI (nan): {'Auroc': np.float64(0.7560042273225958), 'Accuracy': np.float64(0.7019660032976133)}
  IGCI (nan): {'Auroc': np.float64(0.7085030424777897), 'Accuracy': np.float64(0.6532616064772362)}
  lingam (nan): {'Auroc': np.float64(0.5), 'Accuracy': np.float64(0.5052836178358319)}
  anm (nan): {'Auroc': np.float64(0.4205435373120123), 'Accuracy': np.float64(0.3957878221928985)}
  pnl (nan): {'Auroc': np.float64(0.4127416386491891), 'Accuracy': np.float64(0.4448111308557637)}
  emd (nan): {'Auroc': np.float64(0.726504454822172), 'Accuracy': np.float64(0.6453546816839358)}
  cgnn (nan): {'Auroc': np.float64(0.6625114150297617), 'Accuracy': np.float64(0.6086532864590142)}
  qcd_functionR (nan): {'Auroc': np.float64(0.7302290242625532), 'Accuracy': np.float64(0.699565874059885)}

--- Average Differences from Tuebingen Baseline ---

Dataset: CE-Cha
    M

In [71]:
def analyse_exchangeable_dataset(method):
    acc_weights, labels=synth_k.load_weights(filename='data/dataset_weights.csv', metric='Accuracy')
    aur_weights, labels=synth_k.load_weights(filename='data/dataset_weights.csv', metric='Auroc')
    method=method+"_"
    full_scores=[]
    aur_finalw=[]
    acc_finalw=[]
    in_auroc=[]
    in_auroc_weights=[]
    if method.startswith('funcR_'):
        return np.nan
    for aur_w, acc_w,label in zip(aur_weights, acc_weights, labels):
        file_path = f"predictions/{method}/{label}.txt"
        if not os.path.exists(file_path):
            continue
        data = np.loadtxt(file_path)
        # Split the data into scores and weights
        scores = data[:, 0]  # First column
        full_scores.extend(scores)
        weights = data[:, 1]  # Second column
        #print(weights)
        aur_finalw.extend(aur_w*weights)
        acc_finalw.extend(acc_w*weights)
        #AUROC-in
        y_scores, y_true = ce.switch_signs(scores)
        normalized_y=ce.minmax_scale(y_scores)
        normalized_y[np.isnan(normalized_y)] = 0.5
        auroc = ce.roc_auc_score(y_true,normalized_y,sample_weight=weights)
        in_auroc.append(auroc)
        in_auroc_weights.append(aur_w)
    full_scores = np.array(full_scores)
    aur_finalw = np.array(aur_finalw)
    acc_finalw = np.array(acc_finalw)
    #in-AUROC
    in_auroc = np.array(in_auroc)
    in_auroc_weights = np.array(in_auroc_weights)
    in_auroc = sum(in_auroc * in_auroc_weights) / sum(in_auroc_weights)
    #AUROC
    y_scores, y_true = ce.switch_signs(full_scores)
    normalized_y=ce.minmax_scale(y_scores)
    normalized_y[np.isnan(normalized_y)] = 0.5
    auroc = ce.roc_auc_score(y_true,normalized_y,sample_weight=aur_finalw)
    #Accuracy
    guess=ce.sign_to_binary(full_scores)
    accuracy = sum(guess*acc_finalw)/sum(acc_finalw)
    return auroc, accuracy


def test_exchangeable_synthetic(method):
    _, labels=synth_k.load_weights(filename='data/dataset_weights.csv', metric='Accuracy')
    for dataset in labels:
        run_synthetic(method,dataset)
        analyse_predictions(method.__name__,dataset)
    return analyse_exchangeable_dataset(method.__name__)
    

In [70]:
methods=["anm", "cgnn", "emd", "IGCI", "lingam", "pnl", "qcd_functionR", "RECI", "SlopeR"]
for method in methods:
    auroc,accuracy = analyse_exchangeable_dataset(method)
    print(f"Method: {method}, AUROC: {auroc}, Accuracy: {accuracy}")

Method: anm, AUROC: 0.4443770814706875, Accuracy: 0.4097162467569817
Method: cgnn, AUROC: 0.6899586954851399, Accuracy: 0.6156146241905763
Method: emd, AUROC: 0.7970881738778881, Accuracy: 0.6434185516073664
Method: IGCI, AUROC: 0.817996482143, Accuracy: 0.6905381419004423
Method: lingam, AUROC: 0.4933968516176501, Accuracy: 0.5107275549387218
Method: pnl, AUROC: 0.47119538530971616, Accuracy: 0.48255985629036086
Method: qcd_functionR, AUROC: 0.6159861418345791, Accuracy: 0.6028731759614974
Method: RECI, AUROC: 0.796919939224272, Accuracy: 0.709471926933653
Method: SlopeR, AUROC: 0.16836376562858105, Accuracy: 0.29031221858038175
